# Camada Gold — Modelagem Dimensional

Este notebook transforma as tabelas tratadas da Silver em uma modelagem dimensional destinada ao consumo analítico.

São construídas:

- dimensões de filmes, gêneros, pessoas, produtoras e avaliações;
- tabela fato de desempenho dos filmes;
- tabelas-ponte para relacionamentos muitos-para-muitos.

As chaves substitutas (`surrogate keys`) são geradas com `row_number()`.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalog = "olist_medallion"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

## Dimensão de filmes — `gold.dim_movies`

A dimensão contém os principais atributos descritivos de cada filme.

Uma chave substituta `sk_movie_id` é criada independentemente da chave natural `id_filme`, seguindo o modelo dimensional solicitado.

In [0]:
df_info = spark.table(f"{catalog}.silver.tb_info_filmes")
df_dim_movies = df_info.select(
    "id_filme", "titulo", "data_lancamento", "ano_lancamento",
    "duracao_minutos", "idioma_original", "status_filme", "sinopse"
    # Gera uma surrogate key numérica e independente do identificador da origem.
).withColumn("sk_movie_id", F.row_number().over(Window.orderBy("id_filme")).cast("bigint"))
df_dim_movies.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{catalog}.gold.dim_movies"
)
df_dim_movies = spark.table(f"{catalog}.gold.dim_movies")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


## Dimensões de gêneros, pessoas e produtoras

Cada dimensão contém apenas entidades distintas.

Pessoas físicas incluem os tipos Ator, Diretor e Roteirista, enquanto produtoras são separadas em uma dimensão própria. Cada dimensão recebe uma chave substituta.

In [0]:
df_generos = spark.table(f"{catalog}.silver.tb_generos")
df_entidades = spark.table(f"{catalog}.silver.tb_pessoas_empresas")

df_dim_genres = df_generos.select("nome_genero").dropDuplicates().withColumn(
    "sk_genre_id", F.row_number().over(Window.orderBy("nome_genero")).cast("bigint")
)
df_dim_genres.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{catalog}.gold.dim_genres"
)
df_dim_genres = spark.table(f"{catalog}.gold.dim_genres")

# Produtoras são excluídas desta dimensão porque possuem uma dimensão própria.
df_dim_people = df_entidades.filter(
    "tipo_entidade IN ('Ator', 'Diretor', 'Roteirista')"
).select(
    F.col("nome_entidade").alias("nome_pessoa"),
    F.col("tipo_entidade").alias("tipo_pessoa")
).dropDuplicates().withColumn("sk_person_id", F.row_number().over(Window.orderBy("nome_pessoa", "tipo_pessoa")).cast("bigint"))
df_dim_people.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{catalog}.gold.dim_people"
)
df_dim_people = spark.table(f"{catalog}.gold.dim_people")

# Apenas entidades classificadas como Produtora compõem dim_companies.
df_dim_companies = df_entidades.filter("tipo_entidade = 'Produtora'").select(
    F.col("nome_entidade").alias("nome_produtora")
).dropDuplicates().withColumn("sk_company_id", F.row_number().over(Window.orderBy("nome_produtora")).cast("bigint"))
df_dim_companies.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{catalog}.gold.dim_companies"
)
df_dim_companies = spark.table(f"{catalog}.gold.dim_companies")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


## Tabelas-ponte

Filmes possuem relacionamentos muitos-para-muitos com gêneros, pessoas e produtoras.

Por esse motivo, esses relacionamentos não são adicionados diretamente à tabela fato. São utilizadas tabelas-ponte contendo somente as chaves substitutas das entidades relacionadas, evitando a duplicação do grão da fato.

In [0]:
# Relação muitos-para-muitos entre filmes e gêneros.
df_bridge_genre = df_generos.join(
    df_dim_movies.select("id_filme", "sk_movie_id"), on="id_filme", how="inner"
).join(df_dim_genres, on="nome_genero", how="inner").select(
    "sk_movie_id", "sk_genre_id"
).dropDuplicates()
df_bridge_genre.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{catalog}.gold.bridge_movie_genre"
)

df_people_filme = df_entidades.filter(
    "tipo_entidade IN ('Ator', 'Diretor', 'Roteirista')"
).select(
    "id_filme", F.col("nome_entidade").alias("nome_pessoa"),
    F.col("tipo_entidade").alias("tipo_pessoa")
)
# Relação muitos-para-muitos entre filmes e pessoas.
df_bridge_person = df_people_filme.join(
    df_dim_movies.select("id_filme", "sk_movie_id"), on="id_filme", how="inner"
).join(df_dim_people, on=["nome_pessoa", "tipo_pessoa"], how="inner").select(
    "sk_movie_id", "sk_person_id"
).dropDuplicates()
df_bridge_person.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{catalog}.gold.bridge_movie_person"
)
# Relação muitos-para-muitos entre filmes e produtoras.
df_companies_filme = df_entidades.filter("tipo_entidade = 'Produtora'").select(
    "id_filme", F.col("nome_entidade").alias("nome_produtora")
)
df_bridge_company = df_companies_filme.join(
    df_dim_movies.select("id_filme", "sk_movie_id"), on="id_filme", how="inner"
).join(df_dim_companies, on="nome_produtora", how="inner").select(
    "sk_movie_id", "sk_company_id"
).dropDuplicates()
df_bridge_company.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{catalog}.gold.bridge_movie_company"
)

## Tabela fato — `gold.fact_movies_performance`

A tabela fato possui grão de **um registro por filme lançado**.

As métricas financeiras e de engajamento são consolidadas utilizando `id_filme` somente durante os joins e mantendo `sk_movie_id` como chave estrangeira da dimensão de filmes.

Os joins são realizados sobre tabelas previamente deduplicadas para evitar alteração do grão da fato.

In [0]:
df_fin = spark.table(f"{catalog}.silver.tb_financeiro_filmes")
df_met = spark.table(f"{catalog}.silver.tb_metricas_engajamento")

# Regra de negócio: somente filmes com status "Lançado"
# participam da tabela fato de desempenho.
df_lancados = df_dim_movies.filter(
    F.col("status_filme") == "Lançado"
)

# LEFT JOIN preserva o filme mesmo quando alguma métrica financeira
# ou de engajamento está ausente.
df_fato = df_lancados.select("id_filme", "sk_movie_id").join(
    df_fin, on="id_filme", how="left"
).join(df_met, on="id_filme", how="left").select(
    "sk_movie_id", "orcamento_usd", "receita_usd", "lucro_usd",
    "orcamento_brl", "receita_brl", "lucro_brl",
    "popularidade", "nota_media_tmdb", "qtd_votos_tmdb",
    "nota_media_imdb", "qtd_votos_imdb"
)

# A comparação entre filmes elegíveis e linhas da fato auxilia
# na validação de que os joins não multiplicaram o grão.
print("Filmes elegíveis:", df_lancados.count())
print("Linhas da fato:", df_fato.count())
display(df_fato.limit(10))

Filmes elegíveis: 96463
Linhas da fato: 96463


sk_movie_id,orcamento_usd,receita_usd,lucro_usd,orcamento_brl,receita_brl,lucro_brl,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
1,null,null,null,null,null,null,1.132,0.0,0,6.8,27
2,null,null,null,null,null,null,0.6,0.0,0,null,40
3,null,null,null,null,null,null,0.6,0.0,0,4.7,10
4,null,null,null,null,null,null,1.169,0.0,0,4.8,16
5,null,null,null,null,null,null,0.6,0.0,0,7.2,15
6,null,null,null,null,null,null,0.615,0.0,0,null,25
7,null,null,null,null,null,null,0.6,0.0,0,5.9,10
8,4700000.00,null,null,29100990.00,null,null,1.489,6.75,6,6.2,382
9,null,null,null,null,null,null,0.6,0.0,0,7.7,25
10,6000000.00,null,null,37150200.00,null,null,13.212,6.8,15,null,236


### Persistência da tabela fato

Após a validação do grão, a tabela fato é persistida em formato Delta.

In [0]:
df_fato.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{catalog}.gold.fact_movies_performance"
)

## Dimensão de avaliações — `gold.dim_reviews`

As avaliações individuais da Silver são agregadas por filme.

Para cada filme são calculadas:

- quantidade de avaliações;
- nota média dos usuários, arredondada para duas casas decimais.

Filmes sem avaliações recebem quantidade igual a zero, enquanto a média permanece ausente.

In [0]:
df_reviews_resumo = (
    spark.table(f"{catalog}.silver.tb_avaliacoes_usuarios")
    # Consolida várias avaliações individuais em uma única linha por filme.
    .groupBy("id_filme")
    .agg(
        F.count("*").cast("int").alias("qtd_avaliacoes_usuarios"),
        F.round(F.avg("nota_usuario"), 2).cast("double").alias("nota_media_usuarios")
    )
)

# O LEFT JOIN garante que filmes sem avaliações também estejam presentes
# na dimensão, com quantidade de avaliações igual a zero.
df_dim_reviews = df_dim_movies.select("id_filme", "sk_movie_id").join(
    df_reviews_resumo, on="id_filme", how="left"
).withColumn(
    "qtd_avaliacoes_usuarios",
    F.when(F.col("qtd_avaliacoes_usuarios").isNull(), 0)
     .otherwise(F.col("qtd_avaliacoes_usuarios")).cast("int")
).withColumn("sk_review_id", F.row_number().over(Window.orderBy("sk_movie_id")).cast("bigint")).select(
    "sk_review_id", "sk_movie_id", "qtd_avaliacoes_usuarios", "nota_media_usuarios"
)
df_dim_reviews.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{catalog}.gold.dim_reviews"
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
